In [0]:
# 1. Definir los widgets primero
dbutils.widgets.removeAll()
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Ambiente")
dbutils.widgets.text("process_date", "", "Fecha Proceso (DDMMYYYY)")

# 2. Ahora sí puedes capturar los valores
env = dbutils.widgets.get("env")
process_date = dbutils.widgets.get("process_date")

catalog = f"{env}_fraud"
gold = f"{catalog}.gold"

# El resto de tu código...
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")

In [0]:
from pyspark.sql.functions import *

# Parámetros
env = dbutils.widgets.get("env")
catalog = f"{env}_fraud"
process_date = dbutils.widgets.get("process_date")

df_trans = spark.table(f"{catalog}.silver.transacciones_silver").filter(col("process_date") == process_date)
df_cust  = spark.table(f"{catalog}.silver.clientes_silver")
df_cards = spark.table(f"{catalog}.silver.tarjetas_silver")
df_mcc   = spark.table(f"{catalog}.silver.mcc_silver")


df_gold_master = df_trans.alias("t") \
    .join(df_cust.alias("c"), col("t.id_cliente") == col("c.id_cliente"), "left") \
    .join(df_cards.alias("cr"), col("t.id_tarjeta") == col("cr.id_tarjeta"), "left") \
    .join(df_mcc.alias("m"), col("t.categoria_mcc") == col("m.codigo_mcc"), "left") \
    .select(
        col("t.fecha_completa"),
        col("t.id_cliente"),
        col("c.nombre").alias("nombre_cliente"),
        col("c.rango_fico"),
        col("c.estado").alias("estado_cliente"),
        col("cr.tipo_tarjeta"),
        col("cr.limite_credito"),
        col("t.monto"),
        col("t.categoria_mcc"),
        col("m.categoria_grupo"),
        col("t.es_fraude").cast("int"),
        col("t.hora_pico"),
        col("t.dia_semana"),
        current_timestamp().alias("ingestion_timestamp")
    )

(df_gold_master.write.format("delta")
 .mode("append") # Append porque queremos el historial para el dashboard
 .saveAsTable(f"{catalog}.gold.fact_fraud_analysis"))

df_daily = df_gold_master.groupBy(to_date(col("fecha_completa")).alias("fecha")) \
    .agg(
        count("*").alias("total_transacciones"),
        sum("monto").alias("monto_total"),
        sum("es_fraude").alias("cantidad_fraudes"),
        (sum("es_fraude") / count("*") * 100).alias("tasa_fraude_pct")
    ).withColumn("ingestion_timestamp", current_timestamp())

(df_daily.write.format("delta")
 .mode("overwrite") # Overwrite porque recalculamos el resumen diario
 .saveAsTable(f"{catalog}.gold.agg_daily_fraud_metrics"))

# --- AUDITORÍA FINAL ---
print(f"🚀 Capa Gold procesada.")
print(f"Registros en Fact Master: {df_gold_master.count()}")
display(spark.table(f"{catalog}.gold.fact_fraud_analysis").limit(5))